In [ ]:
from src.graph_to_vec_converter import  HVs
from sklearn.metrics       import accuracy_score, classification_report
from sklearn.linear_model  import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from src.graph_generation import NMNISTGraphDataset
from src.loader import ev_loader
from src.graphcnnVSA_Binding_FULL import GraphCNN
from src.codebook import CodeBook
import torch
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing   import StandardScaler
from sklearn.svm             import SVC
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import accuracy_score
from torch.utils.data import ConcatDataset
import numpy as np

print("[LOG] - parameter initialization.")
# GRAPH parameters
DATA_NAME = "NCARS" # NCARS, NMNIST
DATA_PATH = "data"
DATASET = "full"  # full / test      size of dataset loading for training and testing

# if DATA_NAME == "NCARS":
X_MAX = 360
Y_MAX = 360
T_MAX = 100_000_000
T_STEP = 10_000

NORMALIZE_FEAT = False
NUM_OF_GRAPH_EVENTS = 100  # None, 10, 50, 100. etc
R = 7
D_MAX = 30

# NOISE parameters
NOICE_REMOVED = True
NR_BIN_XY_SIZE = 4
NR_TIME_BIN_SIZE = 20_000
NR_MINIMUM_EVENTS = 3

# GVFA parameters
HV_DIMENTION = 1000
LAYERS = 5
DELTA = 1  # 2
EQUATION = 11
DEVICE = torch.device("cpu")

# load event streams
print("[LOG] - Loading events")
# full_ev_ds = ev_loader(root=DATA_PATH, dataset=DATASET)
ds = ev_loader(root=DATA_PATH, dataset=DATASET)

train_ds, test_ds = train_test_split(ds, test_size=0.2, random_state=10, shuffle=True)   
print("[LOG] - Making class objects.")


[LOG] - parameter initialization.
[LOG] - Loading events
LOG: load full dataset


In [20]:
print("nmberofsample = ",len(ds))

print(ds[100][0])
print(ds[100][0][-1][0])
min_evs = len(ds[1][0])
max_evs = len(ds[1][0])

min_time = ds[1][0][-1][0]
max_time = ds[1][0][-1][0]
for i in range(len(ds)):
    # print(len(ds[i][0]))
    if min_evs > len(ds[i][0]):
        min_evs = len(ds[i][0])
    if max_evs < len(ds[i][0]):
        max_evs = len(ds[i][0])

    if min_time > ds[i][0][-1][0]:
        min_time = ds[i][0][-1][0]
    if max_time < ds[i][0][-1][0]:
        max_time = ds[i][0][-1][0]

    

print("Number of events in sample; min:", min_evs)
print("Number of events in sample; max:", max_evs)

print("Sample time range; min:", min_time)
print("Sample time range; max:", max_time)

nmberofsample =  24029
[(    0,  5, 20, 0) (  155, 16, 14, 1) (  178, 31,  0, 0)
 (  218, 31, 11, 0) (  277,  9,  5, 1) (  396, 29, 45, 0)
 (  461, 27, 18, 0) (  725, 17, 36, 0) (  794, 16, 36, 0)
 (  953, 26, 31, 0) ( 1141, 25, 31, 0) ( 1283,  8,  6, 1)
 ( 1756, 20,  9, 1) ( 1793, 34,  1, 0) ( 1933, 18,  9, 1)
 ( 2048, 29, 22, 1) ( 2152, 28, 21, 1) ( 2268, 28, 33, 0)
 ( 2698, 28, 32, 0) ( 3006, 34,  2, 0) ( 3008, 11, 18, 0)
 ( 3094, 34, 13, 1) ( 3120, 18,  3, 1) ( 3266, 34, 26, 1)
 ( 3299, 12, 16, 0) ( 3635, 10, 10, 1) ( 3656, 29,  6, 1)
 ( 3670, 28,  2, 1) ( 3722, 34,  0, 0) ( 3768,  1, 35, 0)
 ( 4049, 26, 18, 0) ( 4199, 11, 26, 0) ( 4415, 14, 17, 0)
 ( 4621, 20, 19, 0) ( 4673, 26, 19, 0) ( 4797, 27, 32, 0)
 ( 5023, 26, 17, 0) ( 5173, 29, 19, 0) ( 5427, 28, 19, 0)
 ( 5547, 20, 31, 0) ( 5582, 32, 35, 1) ( 5786, 11, 10, 1)
 ( 6085, 13,  7, 1) ( 6285, 17, 20, 1) ( 6408,  8, 17, 0)
 ( 6554, 26, 10, 1) ( 6637, 32, 14, 1) ( 6775, 10, 19, 1)
 ( 6967, 29, 21, 1) ( 7727, 26,  2, 1) ( 7890, 27